# Phase 1: Training Pipeline Setup & Model Training

**Task**: Conifer Family Classification (Task A)

This notebook demonstrates the complete training pipeline for family-level classification of 4 conifer families.

## Contents

1. [Setup & Imports](#1-setup)
2. [Verify Dataset](#2-verify-dataset)
3. [Create Dataloaders](#3-create-dataloaders)
4. [Test Model Creation](#4-test-model-creation)
5. [Train Single Model (ResNet-50)](#5-train-single-model)
6. [Evaluate on Test Set](#6-evaluate-on-test-set)
7. [Train All 5 Models](#7-train-all-5-models)

In [ ]:
# 1. Setup & Imports
```python
import sys
sys.path.insert(0, '..')  # Add parent directory to path

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
```

## 2. Verify Dataset

Verify the dataset structure and check class distribution.

In [ ]:
```python
from pathlib import Path
from src.data.dataset import ConiferDataset

# Dataset path
DATA_ROOT = Path('../data/split')

# Check structure
print("Dataset structure:")
for split in ['train', 'val', 'test']:
    split_path = DATA_ROOT / split
    if split_path.exists():
        families = list(split_path.iterdir())
        print(f"  {split}/: {len(families)} families")

# Load dataset and check distribution
train_dataset = ConiferDataset(DATA_ROOT, split='train', task='taskA')
print(f"\nTotal training samples: {len(train_dataset)}")
print(f"Classes: {train_dataset.classes}")
print(f"\nClass distribution:")
for family, count in train_dataset.get_class_distribution().items():
    print(f"  {family}: {count} ({count/len(train_dataset)*100:.1f}%)")
```

## 3. Create Dataloaders

Create train/val/test dataloaders with appropriate transforms.

In [ ]:
```python
from src.data import get_dataloaders, get_train_transforms, get_val_transforms

# Configuration
BATCH_SIZE = 64
IMG_SIZE = 224
NUM_WORKERS = 4

# Create dataloaders
train_loader, val_loader, test_loader = get_dataloaders(
    data_root=DATA_ROOT,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    img_size=IMG_SIZE,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

# Verify a batch
images, labels = next(iter(train_loader))
print(f"\nBatch shape: {images.shape}")
print(f"Labels shape: {labels.shape}")
```

## 4. Test Model Creation

Create a model and verify it works correctly.

In [ ]:
```python
from src.models import create_model, BACKBONE_NAMES

# Check available backbones
print("Available backbones:")
for name, timm_name in BACKBONE_NAMES.items():
    print(f"  {name}: {timm_name}")

# Create a model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = create_model('resnet50', num_classes=4, pretrained=True, freeze_backbone=True)
model = model.to(device)

# Test forward pass
images = images.to(device)
outputs = model(images)
print(f"\nOutput shape: {outputs.shape}")
print(f"Expected: ({images.shape[0]}, 4)")

# Verify gradients
print(f"\nBackbone frozen: {not any(p.requires_grad for n, p in model.named_parameters() if 'head' not in n)}")
print(f"Head trainable: {all(p.requires_grad for n, p in model.named_parameters() if 'head' in n)}")
```

## 5. Train Single Model (ResNet-50)

Train ResNet-50 with frozen backbone for 50 epochs.

In [ ]:
```python
from src.train import TrainingEngine, create_optimizer, create_scheduler, create_criterion
from src.utils import set_seed

# Configuration
EPOCHS = 50
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
SEED = 42

# Set seed
set_seed(SEED)

# Create model
model = create_model('resnet50', num_classes=4, pretrained=True, freeze_backbone=True)
model = model.to(device)

# Create optimizer
optimizer = create_optimizer(model, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Create scheduler
scheduler = create_scheduler(optimizer, epochs=EPOCHS, scheduler_type='cosine')

# Create criterion
criterion = create_criterion(label_smoothing=LABEL_SMOOTHING)

# Create training engine
engine = TrainingEngine(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    device=device,
    use_amp=True,  # Mixed precision
)

# Train
print("Starting training...")
results = engine.fit(
    epochs=EPOCHS,
    output_dir=Path('../experiments/taskA/resnet50'),
    name='resnet50',
    save_best_only=False,
    verbose=True,
)

print(f"\nTraining complete!")
print(f"Best val accuracy: {results['best_val_acc']:.2f}%")
```

## 6. Evaluate on Test Set

Load the best checkpoint and evaluate on the test set.

In [ ]:
```python
from src.eval import compute_metrics, format_metrics, save_confusion_matrix

CLASS_NAMES = ['Cupressaceae', 'Pinaceae', 'Podocarpaceae', 'Taxaceae']

# Load best checkpoint
checkpoint_path = Path('../experiments/taskA/resnet50/resnet50_best.pth')
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])

# Evaluate
print("Evaluating on test set...")
metrics = compute_metrics(
    model=model,
    data_loader=test_loader,
    device=device,
    num_classes=4,
    class_names=CLASS_NAMES,
)

# Print results
print(format_metrics(metrics, CLASS_NAMES))

# Save confusion matrix
save_confusion_matrix(
    cm=metrics['confusion_matrix'],
    class_names=CLASS_NAMES,
    output_path=Path('../experiments/taskA/resnet50/confusion_matrix.png'),
    title='Confusion Matrix - ResNet-50 (Test Set)',
)
print("\nConfusion matrix saved!")
```

## 7. Train All 5 Models

Train all backbone architectures and compare results.

In [ ]:
```python
# Run from command line instead:
# python scripts/train_taskA.py --all_models --epochs 50

# Or train individual models:
# python scripts/train_taskA.py --backbone resnet50 --epochs 50
# python scripts/train_taskA.py --backbone efficientnet_b3 --epochs 50
# python scripts/train_taskA.py --backbone mobilenetv3_large --epochs 50
# python scripts/train_taskA.py --backbone convnext_tiny --epochs 50
# python scripts/train_taskA.py --backbone swin_tiny --epochs 50

# To use 90-degree rotation augmentation:
# python scripts/train_taskA.py --backbone resnet50 --epochs 50 --use_90rotation
```

---

## Quick Start Commands

### 1. Install Dependencies

```bash
pip install -r requirements.txt
```

### 2. Verify Setup

```bash
python -c "import torch; print(f'PyTorch: {torch.__version__}'); print(f'CUDA: {torch.cuda.is_available()}')"
```

### 3. Train Single Model (ResNet-50)

```bash
python scripts/train_taskA.py --backbone resnet50 --epochs 50 --seed 42
```

### 4. Train All Models

```bash
python scripts/train_taskA.py --all_models --epochs 50 --seed 42
```

### 5. Evaluate a Model

```bash
python scripts/eval_taskA.py \
    --checkpoint experiments/taskA/resnet50/resnet50_best.pth \
    --backbone resnet50 \
    --save_cm --save_f1
```

### 6. Train with Different Seeds (for reproducibility)

```bash
python scripts/train_taskA.py --backbone resnet50 --epochs 50 --seed 42
python scripts/train_taskA.py --backbone resnet50 --epochs 50 --seed 123
python scripts/train_taskA.py --backbone resnet50 --epochs 50 --seed 456
```

### 7. Train with 90-degree Rotation Augmentation

```bash
python scripts/train_taskA.py --backbone resnet50 --epochs 50 --use_90rotation
```